In [ ]:
# ── Cell 0: Mount Drive & Unzip Dataset ───────────────────────────────────────
# Upload heritagelens-data.zip (~2540 MB) to Google Drive → MyDrive, then run.
# Extracts to /content/metadata_merged.json + /content/images/<filename>.jpg
# Dataset: 2285 images, trained on Cap 1 (Gemini) only
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import drive
import os, shutil, sys, json
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

ROOT        = Path("/content")
MERGED_PATH = ROOT / "metadata_merged.json"
IMAGES_DIR  = ROOT / "images"

# Skip unzip if already extracted (avoids re-doing 2.5 GB extraction on re-run)
if MERGED_PATH.exists() and IMAGES_DIR.exists() and any(IMAGES_DIR.iterdir()):
    print("Already extracted — skipping unzip.")
else:
    zip_name   = "heritagelens-data.zip"
    candidates = [Path("/content/drive/MyDrive") / zip_name, Path("/content") / zip_name]
    target_zip = next((p for p in candidates if p.exists()), None)

    if target_zip:
        print(f"Found: {target_zip}  ({target_zip.stat().st_size/1e6:.1f} MB)")
        print("Unzipping ...")
        shutil.unpack_archive(str(target_zip), "/content/", "zip")
        print("Done.")
    else:
        raise FileNotFoundError("heritagelens-data.zip not found. Upload to Drive → MyDrive.")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ── Verify ─────────────────────────────────────────────────────────────────────
with open(MERGED_PATH) as f:
    meta = json.load(f)

img_count = sum(1 for f in IMAGES_DIR.iterdir() if f.is_file()) if IMAGES_DIR.exists() else 0

ext_n = sum(1 for e in meta if e.get("image_type") == "exterior")
obj_n = sum(1 for e in meta if e.get("image_type") == "object")

print(f"\nROOT              : {ROOT}")
print(f"metadata entries  : {len(meta)}  (exterior={ext_n}, object={obj_n})")
print(f"images on disk    : {img_count}")
print(f"unique monuments  : {len(set(e['monument_name'] for e in meta))}")

if img_count < len(meta) * 0.95:
    print("WARNING: fewer images on disk than metadata entries — check the zip.")
else:
    print("\n✓ All good — run the next cells in order.")

In [ ]:
# ── Cell 1: Dataset Preview ────────────────────────────────────────────────────
import json, random, unicodedata, textwrap
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

ROOT        = Path("/content")
MERGED_PATH = ROOT / "metadata_merged.json"
IMAGES_DIR  = ROOT / "images"

with open(MERGED_PATH) as f:
    metadata = json.load(f)

cap1_lens = [len(e["captions"][0].split()) for e in metadata]
ext_n = sum(1 for e in metadata if e.get("image_type") == "exterior")
obj_n = sum(1 for e in metadata if e.get("image_type") == "object")
n_monuments = len(set(e['monument_name'] for e in metadata))
print(f"Total images       : {len(metadata)}  (exterior={ext_n}, object={obj_n})")
print(f"Unique monuments   : {n_monuments}")
print(f"Avg Cap1 (Gemini)  : {sum(cap1_lens)/len(cap1_lens):.1f} words  ← training target")

def _nfc(s): return unicodedata.normalize("NFC", s)
_lookup = {}
if IMAGES_DIR.exists():
    for f in IMAGES_DIR.iterdir():
        if f.is_file():
            _lookup[_nfc(f.name)] = f

matched = sum(1 for e in metadata if _nfc(e["image_id"]) in _lookup)
print(f"\nImages on disk     : {len(_lookup)}  |  Matched: {matched}/{len(metadata)}")
if matched < len(metadata) * 0.95:
    print("⚠ Re-upload heritagelens-data.zip (~2540 MB)")
else:
    print("✓ All images found")

# Preview grid — Cap 1 (Gemini) only
samples = random.sample(metadata, min(6, len(metadata)))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, e in zip(axes.flatten(), samples):
    path = _lookup.get(_nfc(e["image_id"]))
    if path:
        ax.imshow(mpimg.imread(str(path)))
    else:
        ax.text(0.5, 0.5, "not found", ha="center", va="center", fontsize=9, color="red")
    ax.axis("off")
    cap = e["captions"][0]
    cap_wrapped = "\n".join(textwrap.wrap(cap, width=50))[:200] + ("..." if len(cap) > 200 else "")
    ax.set_title(f"[{e.get('image_type','?')}] {e['monument_name'][:40]}\n{cap_wrapped}",
                 fontsize=7, loc="left")

plt.suptitle(f"HeritageLens — {len(metadata)} images, {n_monuments} monuments (Cap 1 = Gemini)", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ── Cell 2: Install Libraries & Global Setup ──────────────────────────────────
# Using BLIP (Bootstrapping Language-Image Pre-training) — pre-trained on
# 129M image-caption pairs, fine-tuned here on 2285 Nepali heritage images.
# ──────────────────────────────────────────────────────────────────────────────
!pip install -q transformers>=4.36 torch torchvision tqdm nltk pillow
!pip install -q torchmetrics[multimodal] open_clip_torch   # for CLIPScore in Cell 6/7

import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from tqdm import tqdm

ROOT = Path("/content")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Speed up training for fixed image input size (BLIP processes at 384×384)
torch.backends.cudnn.benchmark = True

print(f"ROOT   = {ROOT}")
print(f"Device = {device}")
if device.type == "cuda":
    print(f"GPU    = {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM   = {vram:.1f} GB")
    if vram < 14:
        print("⚠  Less than 14 GB VRAM — consider setting BATCH_SIZE=32 in Cell 3")

# ── Load BLIP processor ───────────────────────────────────────────────────────
MODEL_ID  = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(MODEL_ID)
print(f"\nBLIP processor loaded  : {MODEL_ID}")
print("Ready — run Cell 3 next.")

In [ ]:
# ── Cell 3: Dataset & DataLoaders (BLIP) ──────────────────────────────────────
import json, random, unicodedata
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

def _nfc(s): return unicodedata.normalize("NFC", s)

_IMAGE_CACHE = {}
def _build_cache(images_dir):
    for f in Path(images_dir).iterdir():
        if f.is_file():
            _IMAGE_CACHE.setdefault(_nfc(f.name), f)

# ── Augmentation ──────────────────────────────────────────────────────────────
# NOTE: No horizontal flip — Gemini captions contain directional info
# ("southern-facing", "eastern side") that becomes wrong after a flip.
_aug = transforms.Compose([
    transforms.RandomApply([transforms.ColorJitter(0.3, 0.3, 0.2, 0.05)], p=0.6),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
    transforms.RandomResizedCrop(384, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
])

def _anonymize(caption: str, monument_name: str) -> str:
    """Replace monument name with 'this monument' so model learns architecture,
    not monument names (which it can't reliably identify at inference time)."""
    if monument_name and monument_name in caption:
        return caption.replace(monument_name, "this monument")
    return caption

class HeritageDataset(Dataset):
    """Returns (PIL_image, caption_str) for BLIP fine-tuning.
    Uses Cap 1 (Gemini) with monument names anonymized."""

    def __init__(self, json_path, images_dir, augment=False):
        if not _IMAGE_CACHE:
            _build_cache(images_dir)
        with open(json_path) as f:
            meta = json.load(f)
        self.augment = augment
        self.entries = []
        skipped = 0
        for e in meta:
            path = _IMAGE_CACHE.get(_nfc(e["image_id"]))
            if path:
                self.entries.append((e, path))
            else:
                skipped += 1
        if skipped:
            print(f"  Skipped {skipped} entries (image not found)")

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry, path = self.entries[idx]
        caption = _anonymize(entry["captions"][0], entry.get("monument_name", ""))
        img = Image.open(path).convert("RGB")
        if self.augment:
            img = _aug(img)
        return img, caption

# ── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE    = 32    # Safe default for T4 15GB at 384×384 with AMP float16
                      # Try 48 if epoch 1 completes without OOM — it will be ~30% faster
MAX_CAP_LEN   = 120   # Gemini captions ~85–95 tokens; 80 was truncating them
VAL_FRACTION  = 0.12
TEST_FRACTION = 0.08
LR            = 2e-5
NUM_EPOCHS    = 20
PATIENCE      = 6
SEED          = 42
BLEU_INTERVAL = 3     # compute BLEU every N epochs (beam search is slow, ~75s)

CKPT_DIR  = ROOT / "outputs/checkpoints"
FIG_DIR   = ROOT / "outputs/figures"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Load dataset ──────────────────────────────────────────────────────────────
print("Loading dataset ...")
full_ds = HeritageDataset(
    json_path  = ROOT / "metadata_merged.json",
    images_dir = ROOT / "images",
    augment    = False,
)
print(f"Valid images: {len(full_ds)}")

# ── Spot-check anonymization ──────────────────────────────────────────────────
print("\nAnonymization check (3 samples):")
for i in random.sample(range(len(full_ds)), 3):
    entry, _ = full_ds.entries[i]
    raw  = entry["captions"][0]
    anon = _anonymize(raw, entry.get("monument_name", ""))
    changed = "✓" if raw != anon else "─"
    print(f"  [{changed}] {anon[:100]}...")

# ── Train / Val / Test split ──────────────────────────────────────────────────
n_test  = max(int(len(full_ds) * TEST_FRACTION), 1)
n_val   = max(int(len(full_ds) * VAL_FRACTION), 1)
n_train = len(full_ds) - n_val - n_test
train_sub, val_sub, test_sub = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED),
)

class AugSubset(Dataset):
    def __init__(self, subset):
        self.subset = subset
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, cap = self.subset[idx]
        return _aug(img), cap

train_ds = AugSubset(train_sub)
val_ds   = val_sub
print(f"\nTrain: {n_train}  |  Val: {n_val}  |  Test (held-out): {n_test}")

# ── Collate ───────────────────────────────────────────────────────────────────
def collate_fn(batch):
    images, captions = zip(*batch)
    enc = processor(
        images=list(images),
        text=list(captions),
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_CAP_LEN,
    )
    labels = enc["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    return enc["pixel_values"], enc["input_ids"], enc["attention_mask"], labels

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=4, pin_memory=True,
                          persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=4, pin_memory=True,
                          persistent_workers=True)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")
print(f"Batch size: {BATCH_SIZE}  |  MAX_CAP_LEN: {MAX_CAP_LEN}  |  LR: {LR}")
print("✓ DataLoaders ready — run Cell 4 next.")

In [ ]:
# ── Cell 4: Load BLIP + Optimizer + Scheduler ─────────────────────────────────
# BLIP-base: ViT-B/16 vision encoder + BERT-based text decoder
# Pre-trained on 129M pairs → fine-tuned on 2285 Nepal heritage images (Gemini captions)
# ──────────────────────────────────────────────────────────────────────────────
from transformers import BlipForConditionalGeneration
import torch

MODEL_ID = "Salesforce/blip-image-captioning-base"
model    = BlipForConditionalGeneration.from_pretrained(MODEL_ID).to(device)

# ── Freeze the vision encoder — only fine-tune the text decoder ───────────────
# With 2285 images, we still freeze ViT to avoid overfitting; decoder learns domain text.
# The vision encoder already generalises well; we only need domain-specific text.
for param in model.vision_model.parameters():
    param.requires_grad = False

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_p    = total_p - trainable_p

print(f"Model        : {MODEL_ID}")
print(f"Total params : {total_p/1e6:.1f}M")
print(f"Trainable    : {trainable_p/1e6:.1f}M  (text decoder + cross-attention)")
print(f"Frozen       : {frozen_p/1e6:.1f}M  (ViT vision encoder)")

# ── Optimizer ─────────────────────────────────────────────────────────────────
# Low LR for fine-tuning; slightly higher for the projection layer
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01,
)

# Warmup 2 epochs then cosine decay
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
warmup    = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=2)
cosine    = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - 2, eta_min=1e-7)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[2])

scaler = torch.amp.GradScaler("cuda")

print(f"\nOptimizer : AdamW  LR={LR}  weight_decay=0.01")
print(f"Scheduler : LinearWarmup(2) → CosineAnnealing")
print(f"Scaler    : AMP mixed precision")
print(f"\n✓ Model ready — run Cell 5 (Training) next.")

In [ ]:
# ── Cell 5: Fine-tuning Loop ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import nltk, random as _rnd
nltk.download("punkt_tab", quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ── Generation helper — conditional on domain prefix ─────────────────────────
@torch.no_grad()
def _generate(pil_image, max_new_tokens=100, num_beams=4):
    """Conditional beam search with domain prefix.
    Uses max_new_tokens (not max_length) so prefix tokens don't eat into budget."""
    model.eval()
    inputs = processor(
        images=pil_image,
        text="a nepali heritage monument",
        return_tensors="pt",
    ).to(device)
    ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )
    return processor.decode(ids[0], skip_special_tokens=True)

# ── Training setup ────────────────────────────────────────────────────────────
MAX_GRAD_NORM  = 1.0
BLEU_SAMPLE_N  = 50
BEST_CKPT_PATH = CKPT_DIR / "best_model.pt"   # fast state_dict save during training
history        = {"train_loss": [], "val_loss": [], "bleu": [], "lr": []}
best_val_loss  = float("inf")
patience_ctr   = 0
smoother       = SmoothingFunction().method1

gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print(f"Fine-tuning BLIP on {gpu_name}")
print(f"  Epochs={NUM_EPOCHS}  batch={BATCH_SIZE}  LR={LR}  patience={PATIENCE}")
print(f"  Train batches/epoch: {len(train_loader)}  |  Val batches: {len(val_loader)}")
print(f"  BLEU computed every {BLEU_INTERVAL} epochs")
print("-" * 60)

for epoch in range(NUM_EPOCHS):
    lr_now = optimizer.param_groups[0]["lr"]
    history["lr"].append(lr_now)

    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    total_train = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}/{NUM_EPOCHS} [train]")
    for pixel_values, input_ids, attention_mask, labels in pbar:
        pixel_values   = pixel_values.to(device, non_blocking=True)
        input_ids      = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels         = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            out  = model(pixel_values=pixel_values,
                         input_ids=input_ids,
                         attention_mask=attention_mask,
                         labels=labels)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        total_train += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr_now:.1e}")

    avg_train = total_train / len(train_loader)
    history["train_loss"].append(avg_train)

    # ── Validate (loss) ────────────────────────────────────────────────────────
    model.eval()
    total_val = 0.0
    with torch.no_grad():
        for pixel_values, input_ids, attention_mask, labels in val_loader:
            pixel_values   = pixel_values.to(device, non_blocking=True)
            input_ids      = input_ids.to(device, non_blocking=True)
            attention_mask = attention_mask.to(device, non_blocking=True)
            labels         = labels.to(device, non_blocking=True)
            with torch.amp.autocast("cuda"):
                out = model(pixel_values=pixel_values,
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            total_val += out.loss.item()
    avg_val = total_val / len(val_loader)
    history["val_loss"].append(avg_val)

    # ── BLEU every BLEU_INTERVAL epochs (beam search is expensive) ────────────
    if (epoch + 1) % BLEU_INTERVAL == 0 or epoch == 0:
        bleu_scores = []
        sample_idx  = _rnd.sample(range(len(val_sub)), min(BLEU_SAMPLE_N, len(val_sub)))
        for si in sample_idx:
            entry, path = full_ds.entries[val_sub.indices[si]]
            ref_anon = _anonymize(entry["captions"][0], entry.get("monument_name", ""))
            pil_img  = Image.open(path).convert("RGB")
            gen      = _generate(pil_img)
            gen_toks = gen.split()
            if gen_toks:
                bleu_scores.append(sentence_bleu(
                    [ref_anon.split()], gen_toks,
                    weights=(0.25, 0.25, 0.25, 0.25),
                    smoothing_function=smoother))
        avg_bleu = float(np.mean(bleu_scores)) if bleu_scores else 0.0
    else:
        avg_bleu = history["bleu"][-1] if history["bleu"] else 0.0  # carry forward
    history["bleu"].append(avg_bleu)

    bleu_tag = f"BLEU-4={avg_bleu:.4f}" if (epoch + 1) % BLEU_INTERVAL == 0 or epoch == 0 else "BLEU-4=--"
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS}  lr={lr_now:.1e}  "
          f"train={avg_train:.4f}  val={avg_val:.4f}  {bleu_tag}")

    # ── Checkpoint + early stopping ────────────────────────────────────────────
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        patience_ctr  = 0
        # Fast state_dict save (~0.5s) instead of save_pretrained (~3s per save)
        torch.save(model.state_dict(), BEST_CKPT_PATH)
        print(f"  → Best checkpoint saved  (val={best_val_loss:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    scheduler.step()

# ── Load best weights and save full HF model (once, at end) ──────────────────
print("\nLoading best weights ...")
model.load_state_dict(torch.load(BEST_CKPT_PATH, map_location=device))
model.save_pretrained(CKPT_DIR / "best_model")
processor.save_pretrained(CKPT_DIR / "best_model")
print(f"Best model saved to: {CKPT_DIR / 'best_model'}")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ep = range(1, len(history["train_loss"]) + 1)
axes[0].plot(ep, history["train_loss"], label="Train")
axes[0].plot(ep, history["val_loss"],   label="Val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(ep, history["bleu"], color="green", marker="o", markersize=3)
axes[1].set_title(f"BLEU-4 (val, every {BLEU_INTERVAL} epochs)")
axes[2].plot(ep, history["lr"], color="orange"); axes[2].set_title("LR")
plt.tight_layout()
plt.savefig(FIG_DIR / "training_curves.png", dpi=150); plt.show()
print(f"Done. Best val loss: {best_val_loss:.4f}")
print("Run Cell 6 for final evaluation on the held-out test set.")

In [ ]:
# ── Cell 6: Inference & Evaluation (Fine-tuned BLIP) ─────────────────────────
# Evaluates on the HELD-OUT test set (never seen during training or early stopping)
# References: Cap 1 only, monument names anonymized (consistent with training targets)
import nltk
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import random as _rnd
import textwrap
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

# ── Load best checkpoint ───────────────────────────────────────────────────────
best_dir = CKPT_DIR / "best_model"
if best_dir.exists():
    model     = BlipForConditionalGeneration.from_pretrained(best_dir).to(device)
    processor = BlipProcessor.from_pretrained(best_dir)
    print(f"Loaded best model from: {best_dir}")
else:
    print("No checkpoint found — using current model weights.")
model.eval()

# ── Caption generation — conditional on domain prefix ────────────────────────
@torch.no_grad()
def generate_caption(pil_image, mdl=None, proc=None, num_beams=4, conditional=True):
    mdl  = mdl  or model
    proc = proc or processor
    kwargs = dict(images=pil_image, return_tensors="pt")
    if conditional:
        kwargs["text"] = "a nepali heritage monument"
    inputs = proc(**kwargs).to(device)
    ids = mdl.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=num_beams,
        length_penalty=1.0,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )
    return proc.decode(ids[0], skip_special_tokens=True)

# ── Metric helper — Cap 1 only, anonymized references ────────────────────────
def compute_all_metrics(entries_list, gen_list):
    smoother = SmoothingFunction().method1
    b1, b2, b3, b4, met = [], [], [], [], []
    for entry, gen in zip(entries_list, gen_list):
        raw_ref  = entry["captions"][0]
        ref      = _anonymize(raw_ref, entry.get("monument_name", ""))
        refs     = [ref.split()]
        gen_toks = gen.split()
        if not gen_toks:
            continue
        b1.append(sentence_bleu(refs, gen_toks, weights=(1,0,0,0),             smoothing_function=smoother))
        b2.append(sentence_bleu(refs, gen_toks, weights=(0.5,0.5,0,0),         smoothing_function=smoother))
        b3.append(sentence_bleu(refs, gen_toks, weights=(1/3,1/3,1/3,0),       smoothing_function=smoother))
        b4.append(sentence_bleu(refs, gen_toks, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoother))
        met.append(meteor_score([ref.split()], gen_toks))
    return {
        "bleu1":  float(np.mean(b1))   if b1  else 0.0,
        "bleu2":  float(np.mean(b2))   if b2  else 0.0,
        "bleu3":  float(np.mean(b3))   if b3  else 0.0,
        "bleu4":  float(np.mean(b4))   if b4  else 0.0,
        "meteor": float(np.mean(met))  if met else 0.0,
        "n":      len(b1),
    }

# ── CLIPScore helper — computes cosine similarity between image and caption ───
def compute_clip_scores(pil_images, captions, batch_size=32):
    """
    Manually compute CLIPScore (image↔caption cosine similarity × 100).
    More reliable than torchmetrics wrapper which has version compat issues.
    CLIP text encoder truncates at 77 tokens — long captions lose their tail,
    which is fine since the key visual terms are usually in the first 77 tokens.
    """
    from transformers import CLIPModel, CLIPProcessor as CLIPProc
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
    clip_proc  = CLIPProc.from_pretrained("openai/clip-vit-base-patch16")
    clip_model.eval()

    scores = []
    for i in tqdm(range(0, len(pil_images), batch_size), desc="CLIPScore"):
        imgs = pil_images[i : i + batch_size]
        caps = captions[i : i + batch_size]
        with torch.no_grad():
            inputs = clip_proc(
                text=caps, images=imgs,
                return_tensors="pt", padding=True,
                truncation=True, max_length=77,
            ).to(device)
            out     = clip_model(**inputs)
            img_emb = out.image_embeds
            txt_emb = out.text_embeds
            img_emb = img_emb / img_emb.norm(p=2, dim=-1, keepdim=True)
            txt_emb = txt_emb / txt_emb.norm(p=2, dim=-1, keepdim=True)
            batch_scores = (img_emb * txt_emb).sum(dim=-1) * 100
            scores.extend(batch_scores.cpu().tolist())

    del clip_model  # free VRAM for Cell 7
    torch.cuda.empty_cache()
    return float(np.mean(scores))

# ── Inference grid: 4 images, full captions visible ──────────────────────────
N_SHOW     = 4
sample_idx = _rnd.sample(list(test_sub.indices), min(N_SHOW, len(test_sub)))

fig, axes = plt.subplots(1, 4, figsize=(26, 10))
for ax, ds_idx in zip(axes, sample_idx):
    entry, path = full_ds.entries[ds_idx]
    gt_cap = _anonymize(entry["captions"][0], entry.get("monument_name", ""))
    img    = Image.open(path).convert("RGB")
    gen    = generate_caption(img)

    ax.imshow(img); ax.axis("off")
    itype  = entry.get("image_type", "")
    title  = (f"[{itype}] {entry['monument_name']}\n\n"
              f"GT:\n" + "\n".join(textwrap.wrap(gt_cap, 38)) +
              f"\n\nGEN:\n" + "\n".join(textwrap.wrap(gen, 38)))
    ax.set_title(title, fontsize=7, loc="left",
                 bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.92))

plt.suptitle("Fine-tuned BLIP — Generated vs Ground Truth (held-out test set)", fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "inference_grid.png", dpi=150, bbox_inches="tight"); plt.show()

# ── Full test set evaluation ───────────────────────────────────────────────────
print(f"\nEvaluating fine-tuned BLIP on held-out test set ({len(test_sub)} images) ...")
ft_entries, ft_gens = [], []
for vi in tqdm(test_sub.indices, desc="Fine-tuned BLIP"):
    entry, path = full_ds.entries[vi]
    ft_entries.append(entry)
    ft_gens.append(generate_caption(Image.open(path).convert("RGB")))

ft_metrics = compute_all_metrics(ft_entries, ft_gens)

print("\n" + "="*55)
print(f"  FINE-TUNED BLIP  —  {ft_metrics['n']} test images")
print("  (Cap 1 refs, monument names anonymized)")
print("="*55)
print(f"  BLEU-1  : {ft_metrics['bleu1']:.4f}")
print(f"  BLEU-2  : {ft_metrics['bleu2']:.4f}")
print(f"  BLEU-3  : {ft_metrics['bleu3']:.4f}")
print(f"  BLEU-4  : {ft_metrics['bleu4']:.4f}")
print(f"  METEOR  : {ft_metrics['meteor']:.4f}")
print("="*55)

import json as _json
(FIG_DIR / "ft_metrics.json").write_text(_json.dumps(ft_metrics, indent=2))

# ── CLIPScore ─────────────────────────────────────────────────────────────────
print("\nComputing CLIPScore ...")
ft_pil_imgs = [Image.open(full_ds.entries[vi][1]).convert("RGB") for vi in test_sub.indices]
ft_metrics["clip_score"] = compute_clip_scores(ft_pil_imgs, ft_gens)
(FIG_DIR / "ft_metrics.json").write_text(_json.dumps(ft_metrics, indent=2))
print(f"  CLIPScore (FT) : {ft_metrics['clip_score']:.2f}  (range ~20–35, higher=better)")

In [ ]:
# ── Cell 7: Zero-shot BLIP Baseline ──────────────────────────────────────────
# Loads pristine blip-image-captioning-base (no fine-tuning) and evaluates
# on the same held-out test set with identical references for a fair comparison.
# Zero-shot uses UNCONDITIONAL generation (no domain prefix) for a fair baseline.
import json as _json

print("Loading zero-shot BLIP (no fine-tuning) ...")
zs_model     = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base").to(device)
zs_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
zs_model.eval()
print("Zero-shot BLIP loaded.")

# ── Evaluate zero-shot on same test set (no domain prefix) ────────────────────
print(f"\nEvaluating zero-shot BLIP on held-out test set ({len(test_sub)} images) ...")
zs_entries, zs_gens = [], []
for vi in tqdm(test_sub.indices, desc="Zero-shot BLIP"):
    entry, path = full_ds.entries[vi]
    zs_entries.append(entry)
    zs_gens.append(generate_caption(
        Image.open(path).convert("RGB"),
        mdl=zs_model, proc=zs_processor, conditional=False,
    ))

zs_metrics = compute_all_metrics(zs_entries, zs_gens)
(FIG_DIR / "zs_metrics.json").write_text(_json.dumps(zs_metrics, indent=2))

# ── CLIPScore for zero-shot ────────────────────────────────────────────────────
print("\nComputing CLIPScore (zero-shot) ...")
zs_pil_imgs = [Image.open(full_ds.entries[vi][1]).convert("RGB") for vi in test_sub.indices]
zs_metrics["clip_score"] = compute_clip_scores(zs_pil_imgs, zs_gens)
(FIG_DIR / "zs_metrics.json").write_text(_json.dumps(zs_metrics, indent=2))

# ── Side-by-side comparison ───────────────────────────────────────────────────
ft_metrics = _json.loads((FIG_DIR / "ft_metrics.json").read_text())

print("\n" + "="*68)
print(f"  COMPARISON  —  {ft_metrics['n']} test images  (Cap 1, names anonymized)")
print("="*68)
print(f"  {'Metric':<12}  {'Zero-shot BLIP':>16}  {'Fine-tuned BLIP':>16}  {'Δ':>8}")
print("-"*68)
for m in ["bleu1", "bleu2", "bleu3", "bleu4", "meteor"]:
    zs    = zs_metrics[m]
    ft    = ft_metrics[m]
    delta = ft - zs
    sign  = "+" if delta >= 0 else ""
    print(f"  {m.upper():<12}  {zs:>16.4f}  {ft:>16.4f}  {sign}{delta:>7.4f}")
if "clip_score" in ft_metrics and "clip_score" in zs_metrics:
    zs = zs_metrics["clip_score"]; ft = ft_metrics["clip_score"]
    delta = ft - zs; sign = "+" if delta >= 0 else ""
    print(f"  {'CLIPScore':<12}  {zs:>16.2f}  {ft:>16.2f}  {sign}{delta:>7.2f}")
print("="*68)

improvement = ft_metrics["bleu4"] / zs_metrics["bleu4"] if zs_metrics["bleu4"] > 0 else float("inf")
print(f"\n  Fine-tuning improved BLEU-4 by {improvement:.1f}× over zero-shot BLIP")
print("\nPaste these numbers into the README metrics table.")

In [ ]:
# ── Cell 8: Qualitative Comparison ───────────────────────────────────────────
# Shows 4 val images with Ground Truth, Fine-tuned BLIP, and Zero-shot BLIP
# captions side-by-side, full captions visible.

import textwrap

N_QUAL   = 4
qual_idx = _rnd.sample(list(val_sub.indices), min(N_QUAL, len(val_sub)))

fig, axes = plt.subplots(1, 4, figsize=(26, 12))
fig.patch.set_facecolor("#FAFAFA")

for ax, ds_idx in zip(axes, qual_idx):
    entry, path = full_ds.entries[ds_idx]
    gt_cap = entry["captions"][0]
    img    = Image.open(path).convert("RGB")

    ft_cap = generate_caption(img, mdl=model,    proc=processor)
    zs_cap = generate_caption(img, mdl=zs_model, proc=zs_processor, conditional=False)

    ax.imshow(img)
    ax.axis("off")

    itype  = entry.get("image_type", "")
    header = f"[{itype}] {entry['monument_name']}"
    gt_w   = "\n".join(textwrap.wrap(gt_cap, width=38))
    ft_w   = "\n".join(textwrap.wrap(ft_cap, width=38))
    zs_w   = "\n".join(textwrap.wrap(zs_cap, width=38))
    title  = f"{header}\n\nGT:\n{gt_w}\n\nFT:\n{ft_w}\n\nZS:\n{zs_w}"
    ax.set_title(title, fontsize=7, loc="left", fontfamily="monospace",
                 bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.88))

plt.suptitle("Qualitative Comparison — Ground Truth vs Fine-tuned vs Zero-shot BLIP",
             fontsize=12, fontweight="bold")
plt.tight_layout()
out_path = FIG_DIR / "qualitative_comparison.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")
print("\nGT = Ground truth (Cap 1, Gemini)  |  FT = Fine-tuned BLIP  |  ZS = Zero-shot BLIP")

In [ ]:
# ── Cell 9: Upload Your Own Image ─────────────────────────────────────────────
# Run Cells 0–6 first (training loads best model; Cell 6 loads it for eval).
# Upload any heritage/temple image to get a fine-tuned caption.

from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import textwrap

uploaded = files.upload()
if not uploaded:
    print("No file uploaded. Click 'Choose Files' and select an image.")
else:
    fname = list(uploaded.keys())[0]
    img   = Image.open(fname).convert("RGB")
    cap  = generate_caption(img)

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img)
    ax.axis("off")
    wrapped = "\n".join(textwrap.wrap(cap, width=50))
    ax.set_title(f"HeritageLens Caption:\n\n{wrapped}", fontsize=12, loc="left",
                 bbox=dict(boxstyle="round,pad=0.5", facecolor="wheat", alpha=0.9))
    plt.tight_layout()
    plt.show()
    print(f"\n📷 {fname}")
    print(f"📝 {cap}")